# Masterclass: Pydantic V2 & Data Validation for GenAI & Agentic AI

Welcome to the comprehensive masterclass on **Pydantic V2**, data validation, schema enforcement, type coercion, serialization, and structured outputs for GenAI and Agentic AI applications.

---

### Table of Contents / Interactive Index

- [1. Without Pydantic vs With Pydantic](#1-without-pydantic-vs-with-pydantic)
  - [1.1 Manual Python Validation without Pydantic](#11-manual-python-validation-without-pydantic)
  - [1.2 Schema Definition with Pydantic BaseModel](#12-schema-definition-with-pydantic-basemodel)
- [2. API & User Registration Validation](#2-api--user-registration-validation)
  - [2.1 Defining UserRegistration with EmailStr](#21-defining-userregistration-with-emailstr)
  - [2.2 Error Diagnostics & Handling with ValidationError](#22-error-diagnostics--handling-with-validationerror)
- [3. Product & E-commerce Validation](#3-product--e-commerce-validation)
  - [3.1 Defining the Product Schema](#31-defining-the-product-schema)
- [4. Banking Transaction Validation](#4-banking-transaction-validation)
  - [4.1 MoneyTransfer Schema & Amount Boundaries](#41-moneytransfer-schema--amount-boundaries)
- [5. LLM Model Configuration Validation](#5-llm-model-configuration-validation)
  - [5.1 Bounding LLM Parameters (Temperature & Timeout)](#51-bounding-llm-parameters-temperature--timeout)
- [6. LLM Structured Output with Pydantic](#6-llm-structured-output-with-pydantic)
  - [6.1 Integrating ChatGroq with PersonDetails Schema](#61-integrating-chatgroq-with-persondetails-schema)
  - [6.2 Agentic Routing Decision Schema](#62-agentic-routing-decision-schema)
- [7. Pydantic Field Types & Nullability](#7-pydantic-field-types--nullability)
  - [7.1 Required, Default, Nullable, and Optional Fields](#71-required-default-nullable-and-optional-fields)
- [8. Model Serialization & Export](#8-model-serialization--export)
  - [8.1 Verifying Model Object Class Type](#81-verifying-model-object-class-type)
  - [8.2 Exporting Model to Dictionary (`model_dump`)](#82-exporting-model-to-dictionary-model_dump)
  - [8.3 Exporting Model to JSON String (`model_dump_json`)](#83-exporting-model-to-json-string-model_dump_json)
  - [8.4 Deserializing JSON String back to Dictionary](#84-deserializing-json-string-back-to-dictionary)
- [9. Model Deserialization & Parsing](#9-model-deserialization--parsing)
  - [9.1 Generating OpenAPI JSON Schema (`model_json_schema`)](#91-generating-openapi-json-schema-model_json_schema)
  - [9.2 Validating Dictionary Payloads (`model_validate`)](#92-validating-dictionary-payloads-model_validate)
  - [9.3 Parsing & Validating Raw JSON Strings (`model_validate_json`)](#93-parsing--validating-raw-json-strings-model_validate_json)
- [10. Model Copying & Updating](#10-model-copying--updating)
  - [10.1 Shallow Copying Models (`model_copy`)](#101-shallow-copying-models-model_copy)
  - [10.2 Copying with Selective Field Updates](#102-copying-with-selective-field-updates)
- [11. Custom Validators](#11-custom-validators)
  - [11.1 Single-Field Validation (`@field_validator`)](#111-single-field-validation-field_validator)
  - [11.2 Multi-Field Validation (`@model_validator`)](#112-multi-field-validation-model_validator)
  - [11.3 Comparison Matrix: `@field_validator` vs `@model_validator`](#113-comparison-matrix-field_validator-vs-model_validator)
- [12. Nested Schemas & Complex Models](#12-nested-schemas--complex-models)
  - [12.1 Defining Nested Address & Company Models](#121-defining-nested-address--company-models)
  - [12.2 Instantiating & Validating Nested Payloads](#122-instantiating--validating-nested-payloads)

---

### Data Validation Flowchart

```text
Incoming Untrusted Data (Dict / JSON / LLM Output)
                      ↓
         Pydantic Schema (BaseModel)
                      ↓
  Validation & Constraint Checks:
  ├── Are required fields present?
  ├── Can data types be safely coerced? (e.g. "49" → 49)
  ├── Do values satisfy range/format rules? (gt, ge, le, EmailStr)
  └── Do custom validators pass? (@field_validator, @model_validator)
                      ↓
     ┌────────────────┴────────────────┐
     ▼                                 ▼
[ Valid Data ]                [ Invalid Data ]
Returns Validated Model       Raises ValidationError
```

---

### Core Concepts Quick Reference

- **Pydantic**: Python library for data validation, parsing, and schema enforcement using standard type annotations.
- **Pydantic AI**: Production-grade Agent framework for building GenAI & Agentic AI applications, built on Pydantic's type safety.
- **`BaseModel`**: The core base class providing validation, automatic type parsing, serialization, JSON Schema generation, and rich error messages.
- **`Field()`**: Function used to add validation constraints (`gt`, `ge`, `lt`, `le`, `min_length`, `max_length`), default values, and field descriptions.
- **Pydantic V1 → V2 Method Migration Summary**:
  - `dict()` → `model_dump()`
  - `json()` → `model_dump_json()`
  - `parse_obj()` → `model_validate()`
  - `parse_raw()` → `model_validate_json()`
  - `schema()` → `model_json_schema()`
  - `copy()` → `model_copy()`
  - `@validator` → `@field_validator`
  - `@root_validator` → `@model_validator`

<a id="1-without-pydantic-vs-with-pydantic"></a>
## 1. Without Pydantic vs With Pydantic

<a id="11-manual-python-validation-without-pydantic"></a>
### 1.1 Manual Python Validation without Pydantic
In traditional Python code, validating user input requires writing repetitive, imperative `if` conditions and `isinstance()` checks for every field. This approach quickly becomes unmaintainable as schemas grow.

In [1]:
# Manual validation function without Pydantic
def create_user(data):
    if "name" not in data:
        raise ValueError("Missing 'name' field")
    
    if not isinstance(data["name"], str):
        raise ValueError("Name must be a string")
    
    if "age" not in data:
        raise ValueError("Age is required")
    
    if not isinstance(data["age"], int):
        raise ValueError("Age must be an integer")
    
    if data["age"] < 18:
        raise ValueError("Age must be greater than or equal to 18")
    
    return {
        "message": "User created successfully",
        "user": data
    }


In [2]:
# Valid user data payload
user_data = {
    "name": "Areeb Ahmad", 
    "age": 35
    }

In [3]:
# Testing successful validation with valid user data
create_user(user_data)

{'message': 'User created successfully',
 'user': {'name': 'Areeb Ahmad', 'age': 35}}

In [4]:
# Invalid user data payload: age under 18
user_data = {
    "name": "Areeb Ahmad", 
    "age": 17
    }

In [5]:
# Testing validation failure: age constraint check (raises ValueError)
create_user(user_data)

ValueError: Age must be greater than or equal to 18

In [6]:
# Invalid user data payload: name is a list instead of a string
user_data = {
    "name": ["Areeb"], 
    "age": 30
    }

In [7]:
# Testing validation failure: type check for name (raises ValueError)
create_user(user_data)

ValueError: Name must be a string

<a id="12-schema-definition-with-pydantic-basemodel"></a>
### 1.2 Schema Definition with Pydantic BaseModel
Pydantic eliminates manual checks by providing declarative schema definitions using standard Python type annotations and `Field()` constraints. It also automatically handles type coercion (e.g. converting numeric strings like `"49"` into integers `49`).

In [8]:
# Importing core Pydantic classes and fields
from pydantic import BaseModel, EmailStr, Field, ValidationError


In [9]:
# Defining a declarative User schema using Pydantic BaseModel
class User(BaseModel):
    name: str = Field(..., description="The name of the user")
    age: int = Field(..., ge=18, description="The age of the user, must be 18 or older")


In [10]:
# Valid instantiation of User model
user = User(name="Areeb Ahmad", age=49)


In [11]:
# Accessing validated attribute: name
user.name


'Areeb Ahmad'

In [12]:
# Accessing validated attribute: age
user.age


49

In [13]:
# Instantiating with invalid data (age=17 < 18) raises Pydantic ValidationError
user = User(name="Areeb", age=17)


ValidationError: 1 validation error for User
age
  Input should be greater than or equal to 18 [type=greater_than_equal, input_value=17, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal

In [14]:
# Demonstrating automatic type coercion: string '49' is safely converted to int 49
User(name="Areeb", age="49")


User(name='Areeb', age=49)

In [15]:
# Failed coercion: non-numeric string 'forthy-nine' cannot be parsed into int (raises ValidationError)
User(name="Areeb", age="forthy-nine")


ValidationError: 1 validation error for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='forthy-nine', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

<a id="2-api--user-registration-validation"></a>
## 2. API & User Registration Validation

<a id="21-defining-userregistration-with-emailstr"></a>
### 2.1 Defining UserRegistration with EmailStr
Validating incoming API signup forms with email validation using Pydantic's built-in `EmailStr` type alongside numeric constraints like `Field(ge=18)`.

In [16]:
# User Registration model with EmailStr validation
class UserRegistration(BaseModel):
    name: str
    email: EmailStr
    age: int = Field(ge=18)

In [17]:
# Valid user registration payload
user = UserRegistration(
    name="Areeb",
    email="areeb@gmail.com",
    age=30
)

In [18]:
# Accessing name attribute
user.name


'Areeb'

In [19]:
# Accessing email attribute
user.email


'areeb@gmail.com'

In [20]:
# Accessing age attribute
user.age


30

In [21]:
# Invalid registration payload: invalid email format without '@' (raises ValidationError)
user = UserRegistration(
    name="Areeb", 
    email="areebahmad", 
    age=19
    )

ValidationError: 1 validation error for UserRegistration
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='areebahmad', input_type=str]

In [22]:
# Multiple validation errors: invalid email AND age < 18 (raises ValidationError with 2 errors)
user = UserRegistration(
    name="Areeb", 
    email="areebahmad", 
    age=16
    )

ValidationError: 2 validation errors for UserRegistration
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='areebahmad', input_type=str]
age
  Input should be greater than or equal to 18 [type=greater_than_equal, input_value=16, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal

<a id="22-error-diagnostics--handling-with-validationerror"></a>
### 2.2 Error Diagnostics & Handling with ValidationError
Pydantic captures all schema violations simultaneously and produces precise, field-specific `ValidationError` messages detailing the exact issue (e.g. `email -> value is not a valid email address`, `age -> Input should be greater than or equal to 18`).

- **Schema Diagnostics**: Identifies all invalid fields in a single pass.
- **`ValidationError`**: Encapsulates detailed error locations, message types, and input values.
- **`try / except` Blocks**: Prevents application crashes and enables clean error handling in web APIs.

In [23]:
# Catching ValidationError gracefully in production code
try:
    user = UserRegistration(
        name="Areeb", 
        email="areebahmad", 
        age=16
    )
except ValidationError as e:
    print(e)

2 validation errors for UserRegistration
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='areebahmad', input_type=str]
age
  Input should be greater than or equal to 18 [type=greater_than_equal, input_value=16, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal


<a id="3-product--e-commerce-validation"></a>
## 3. Product & E-commerce Validation

<a id="31-defining-the-product-schema"></a>
### 3.1 Defining the Product Schema
Enforcing e-commerce catalog constraints such as strictly positive product pricing (`price: float = Field(gt=0)`) and non-negative inventory quantities (`quantity: int = Field(ge=0)`).

In [24]:
# E-commerce Product schema definition
class Product(BaseModel):
    name: str
    price: float = Field(gt=0)
    quantity: int = Field(ge=0)


In [25]:
# Instantiating Product with valid parameters
product = Product(
    name="Laptop",
    price=75000,
    quantity=5
)

In [26]:
# Inspecting validated Product instance
product


Product(name='Laptop', price=75000.0, quantity=5)

In [27]:
# Validation failure: negative price (-500) violates Field(gt=0) constraint
Product(
    name="Laptop",
    price=-500,
    quantity=2
)

ValidationError: 1 validation error for Product
price
  Input should be greater than 0 [type=greater_than, input_value=-500, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

<a id="4-banking-transaction-validation"></a>
## 4. Banking Transaction Validation

<a id="41-moneytransfer-schema--amount-boundaries"></a>
### 4.1 MoneyTransfer Schema & Amount Boundaries
Enforcing banking rules and boundary constraints on financial transfers (e.g. minimum required transfer amount `amount: float = Field(gt=500)`).

In [28]:
# Banking MoneyTransfer schema definition
class MoneyTransfer(BaseModel):
    sender_account: str
    receiver_account: str
    amount: float = Field(gt=500)


In [29]:
# Valid money transfer payload above minimum threshold (5,000,000 > 500)
transfer = MoneyTransfer(
    sender_account="ACC001",
    receiver_account="ACC002",
    amount=5000000
)

In [30]:
# Inspecting validated MoneyTransfer object
transfer


MoneyTransfer(sender_account='ACC001', receiver_account='ACC002', amount=5000000.0)

In [31]:
# Boundary failure 1: negative transfer amount (-5000000)
transfer = MoneyTransfer(
    sender_account="ACC001",
    receiver_account="ACC002",
    amount=-5000000
)

ValidationError: 1 validation error for MoneyTransfer
amount
  Input should be greater than 500 [type=greater_than, input_value=-5000000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

In [32]:
# Boundary failure 2: amount 400 is below minimum threshold of 500
transfer = MoneyTransfer(
    sender_account="ACC001",
    receiver_account="ACC002",
    amount=400
)

ValidationError: 1 validation error for MoneyTransfer
amount
  Input should be greater than 500 [type=greater_than, input_value=400, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

In [33]:
# Type failure: invalid string '40L' cannot be coerced to float
transfer = MoneyTransfer(
    sender_account="ACC001",
    receiver_account="ACC002",
    amount="40L"
)

ValidationError: 1 validation error for MoneyTransfer
amount
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='40L', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing

<a id="5-llm-model-configuration-validation"></a>
## 5. LLM Model Configuration Validation

<a id="51-bounding-llm-parameters-temperature--timeout"></a>
### 5.1 Bounding LLM Parameters (Temperature & Timeout)
Guarding GenAI hyper-parameters at runtime by establishing strict range boundaries on LLM settings (e.g. `temperature` bounded between `0.0` and `2.0`, and `timeout` strictly positive).

In [34]:
# LLM Configuration schema with bounded parameters
class ModelConfig(BaseModel):
    model_name: str
    temperature: float = Field(ge=0, le=2)
    timeout: int = Field(gt=0)


In [35]:
# Valid LLM configuration instantiation
config = ModelConfig(
    model_name="gpt-model", 
    temperature=0.7, 
    timeout=30
    )

In [36]:
# Inspecting validated ModelConfig instance
config


ModelConfig(model_name='gpt-model', temperature=0.7, timeout=30)

In [37]:
# Parameter error: temperature 2.1 exceeds upper bound le=2
config = ModelConfig(
    model_name="gpt-model",
    temperature=2.1,
    timeout=30
)

ValidationError: 1 validation error for ModelConfig
temperature
  Input should be less than or equal to 2 [type=less_than_equal, input_value=2.1, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal

In [38]:
# Parameter error: negative timeout -5 violates gt=0 constraint
config = ModelConfig(
    model_name="gpt-model",
    temperature=0.7,
    timeout=-5
)

ValidationError: 1 validation error for ModelConfig
timeout
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

<a id="6-llm-structured-output-with-pydantic"></a>
## 6. LLM Structured Output with Pydantic (`with_structured_output`)

<a id="61-integrating-chatgroq-with-persondetails-schema"></a>
### 6.1 Integrating ChatGroq with PersonDetails Schema
Using LangChain's `llm.with_structured_output(PersonDetails)` to bind a Pydantic schema to an LLM provider (such as Groq), forcing the model to generate structured JSON payloads that automatically parse into Pydantic model instances.

In [39]:
# Importing ChatGroq integration from LangChain
from langchain_groq import ChatGroq


In [40]:
# Initializing LLM instance with ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

In [41]:
# Unstructured LLM invocation returning raw text response
llm.invoke("What is the capital of France?")

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={'reasoning_content': 'The user asks: "What is the capital of France?" The answer: Paris. Provide concise answer.'}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 78, 'total_tokens': 118, 'completion_time': 0.041478584, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.004322083, 'prompt_tokens_details': None, 'queue_time': 0.327461312, 'total_time': 0.045800667}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_63473661d7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07891-60c5-7b72-b480-a15fa876cd40-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 40, 'total_tokens': 118, 'output_token_details': {'reasoning': 22}})

In [42]:
# Defining target extraction schema with field descriptions for LLM guidance
class PersonDetails(BaseModel):
    name: str = Field(description="Name of the person")
    age: int = Field(description="Age of the person")
    city: str = Field(description="City where the person lives")

In [43]:
# Binding Pydantic schema to LLM for enforced structured output
structured_model =llm.with_structured_output(PersonDetails)

In [44]:
# Executing extraction query: LLM extracts unstructured text into PersonDetails instance
structured_model.invoke("Areeb is 30 years old and lives in delhi")

PersonDetails(name='Areeb', age=30, city='Delhi')

<a id="62-agentic-routing-decision-schema"></a>
### 6.2 Agentic Routing Decision Schema
In multi-agent systems and router patterns, Pydantic schemas combined with Python's `Literal` type are used to enforce deterministic routing choices (`"RAG"`, `"WEB"`, or `"LLM"`) along with explanatory reasoning:

```python
from typing import Literal
from pydantic import BaseModel

class RouteDecision(BaseModel):
    route: Literal["RAG", "WEB", "LLM"]
    reasoning: str
```

<a id="7-pydantic-field-types--nullability"></a>
## 7. Pydantic Field Types & Nullability

<a id="71-required-default-nullable-and-optional-fields"></a>
### 7.1 Required, Default, Nullable, and Optional Fields
Pydantic provides explicit field classification rules based on type hints and default values:

| Classification | Syntax Example | Behavior / Enforcement |
| :--- | :--- | :--- |
| **1. Required Field** | `name: str` | Must be provided during instantiation. Cannot be omitted. |
| **2. Default Field** | `country: str = "India"` | Optional to provide. Uses fallback default value if omitted. |
| **3. Required but Nullable** | `middle_name: str | None` | Must be explicitly passed, but acceptable value can be `None`. |
| **4. Optional + Nullable** | `nickname: str | None = None` | Can be safely omitted (defaults to `None`) or explicitly set to `None`. |

In [45]:
class User(BaseModel):
    # 1. REQUIRED FIELD
    # The value must be provided.
    name: str

    # 2. DEFAULT FIELD
    # If no value is provided, "India" will be used automatically.
    country: str = "India"

    # 3. REQUIRED BUT NULLABLE
    # The field must be provided,
    # but its value can be None.
    middle_name: str | None

    # 4. OPTIONAL-TO-PROVIDE + NULLABLE
    # The field does not have to be provided,
    # and its value can also be None.
    nickname: str | None = None


In [46]:
# Instantiating with all fields explicitly supplied
user = User(
    name="Areeb", 
    country="USA", 
    middle_name="Ahmad", 
    nickname="BlackBadge"
    )

In [47]:
# Inspecting User instance with all custom values
user


User(name='Areeb', country='USA', middle_name='Ahmad', nickname='BlackBadge')

In [48]:
# Instantiating without optional fields (country defaults to 'India', nickname defaults to None)
user = User(
    name="Areeb", 
    middle_name="Ahmad"
    )

In [49]:
# Inspecting User instance with default fallback values
user


User(name='Areeb', country='India', middle_name='Ahmad', nickname=None)

In [50]:
# Supplying None to required nullable field (middle_name=None)
user = User(
    name="Areeb", 
    middle_name=None
    )

In [51]:
# Inspecting User instance with middle_name=None
user


User(name='Areeb', country='India', middle_name=None, nickname=None)

In [52]:
# Model with required fields, constraints (ge=18), and default field (city="Bangalore")
class User(BaseModel):
    name: str
    age: int = Field(ge=18)
    city: str = "Bangalore"


In [53]:
# Instantiating User (city will default to 'Bangalore')
user = User(
    name="Areeb", 
    age=30
    )

In [54]:
# Displaying User instance
user


User(name='Areeb', age=30, city='Bangalore')

In [55]:
# Accessing name attribute
user.name


'Areeb'

In [56]:
# Accessing age attribute
user.age


30

In [57]:
# Accessing city attribute (default value)
user.city


'Bangalore'

<a id="8-model-serialization--export"></a>
## 8. Model Serialization & Export (`model_dump`, `model_dump_json`)

<a id="81-verifying-model-object-class-type"></a>
### 8.1 Verifying Model Object Class Type
Checking the class type of a Pydantic model instance (`type(user)`).

In [58]:
# Checking class type of Pydantic model instance
type(user)


__main__.User

<a id="82-exporting-model-to-dictionary-model_dump"></a>
### 8.2 Exporting Model to Dictionary (`model_dump`)
Converting a Pydantic model instance into a standard Python dictionary using `user.model_dump()` (replaces V1 `user.dict()`).

In [65]:
# Serializing Pydantic model instance to standard Python dictionary
user.model_dump()


{'name': 'Areeb', 'age': 30, 'city': 'Bangalore'}

In [66]:
# Verifying type of exported dictionary
type(user.model_dump())


dict

<a id="83-exporting-model-to-json-string-model_dump_json"></a>
### 8.3 Exporting Model to JSON String (`model_dump_json`)
Serializing a Pydantic model instance into a valid JSON string using `user.model_dump_json()` (replaces V1 `user.json()`).

In [67]:
# Serializing Pydantic model instance directly to JSON formatted string
user.model_dump_json()


'{"name":"Areeb","age":30,"city":"Bangalore"}'

In [68]:
# Verifying type of exported JSON string (str)
type(user.model_dump_json())


str

<a id="84-deserializing-json-string-back-to-dictionary"></a>
### 8.4 Deserializing JSON String back to Dictionary
Converting JSON string output back into a standard Python dictionary using Python's built-in `json.loads()`.

In [69]:
# Deserializing JSON string back into a Python dictionary
import json
json.loads(user.model_dump_json())


{'name': 'Areeb', 'age': 30, 'city': 'Bangalore'}

In [70]:
# Verifying type of deserialized object (dict)
type(json.loads(user.model_dump_json()))


dict

<a id="9-model-deserialization--parsing"></a>
## 9. Model Deserialization & Parsing (`model_validate`, `model_validate_json`)

<a id="91-generating-openapi-json-schema-model_json_schema"></a>
### 9.1 Generating OpenAPI JSON Schema (`model_json_schema`)
Extracting OpenAPI-compatible JSON schema definitions for LLM tool binding, Function Calling APIs, and OpenAPI documentation using `User.model_json_schema()`.

In [71]:
# Defining User model for schema generation and validation
class User(BaseModel):
    name: str
    age: int = Field(ge=18)
    city: str = "Bangalore"


In [72]:
# Extracting OpenAPI JSON Schema representation
User.model_json_schema()


{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'age': {'minimum': 18, 'title': 'Age', 'type': 'integer'},
  'city': {'default': 'Bangalore', 'title': 'City', 'type': 'string'}},
 'required': ['name', 'age'],
 'title': 'User',
 'type': 'object'}

In [73]:
# Instantiating User directly
User(name = "areeb", age = 30, city = "Goa")


User(name='areeb', age=30, city='Goa')

<a id="92-validating-dictionary-payloads-model_validate"></a>
### 9.2 Validating Dictionary Payloads (`model_validate`)
Parsing and validating untrusted Python dictionaries into Pydantic model instances using `User.model_validate(data)` (replaces V1 `User.parse_obj(data)`).

In [74]:
# Raw untrusted dictionary payload (note string '30' for age)
data = {
    "name": "Areeb", 
    "age": "30", 
    "city": "Delhi"
    }

In [75]:
# Validating dictionary and instantiating User model with automatic age coercion ('30' -> 30)
User.model_validate(data)


User(name='Areeb', age=30, city='Delhi')

<a id="93-parsing--validating-raw-json-strings-model_validate_json"></a>
### 9.3 Parsing & Validating Raw JSON Strings (`model_validate_json`)
Directly parsing and validating raw JSON strings into Pydantic model instances using `User.model_validate_json(data)` (replaces V1 `User.parse_raw(data)`).

In [76]:
# Raw JSON string payload
data = """
{
    "name": "Areeb", 
    "age": 30, 
    "city": "Bangalore"
}
"""

In [77]:
# Parsing JSON string and validating into User model instance in one step
User.model_validate_json(data)


User(name='Areeb', age=30, city='Bangalore')

<a id="10-model-copying--updating"></a>
## 10. Model Copying & Updating (`model_copy`)

<a id="101-shallow-copying-models-model_copy"></a>
### 10.1 Shallow Copying Models (`model_copy`)
Creating shallow copies of Pydantic model instances using `user1.model_copy()` (replaces V1 `user1.copy()`).

In [78]:
# Creating initial User instance
user1 = User(
    name="Areeb", 
    age=30
    )

In [79]:
# Creating a shallow copy of user1
user2 = user1.model_copy()


In [80]:
# Displaying copied user2 instance
user2


User(name='Areeb', age=30, city='Bangalore')

<a id="102-copying-with-selective-field-updates"></a>
### 10.2 Copying with Selective Field Updates
Creating a modified model instance while updating specific field values using `user1.model_copy(update={"city": "Mumbai"})`. This maintains immutability patterns.

In [81]:
# Creating copy of user1 while updating city field to 'Mumbai'
user3=user1.model_copy(
    update={
        "city": "Mumbai"
        }
)

In [83]:
# Displaying updated user3 instance
user3


User(name='Areeb', age=30, city='Mumbai')

<a id="11-custom-validators"></a>
## 11. Custom Validators (`@field_validator` vs `@model_validator`)

<a id="111-single-field-validation-field_validator"></a>
### 11.1 Single-Field Validation (`@field_validator`)
Defining custom validation and transformation logic for individual fields using `@field_validator` with `@classmethod` (e.g. verifying name length and transforming to title case).

In [84]:
# Importing custom validator decorators from Pydantic V2
from pydantic import field_validator, model_validator


In [85]:
# Model with custom single-field validator (@field_validator)
class Employee(BaseModel):
    name: str
    age: int
    @field_validator("name")
    @classmethod
    def validate_name(cls, value):
        if len(value.strip()) < 3:
            raise ValueError("Name must contain at least 2 characters")
        return value.title()


In [88]:
# Valid instantiation: name 'Areeb' is transformed to title case 'Areeb'
Employee(name="Areeb", age=30)


Employee(name='Areeb', age=30)

In [89]:
# Validation failure: name 'Ar' has length < 3 (raises ValueError)
Employee(name="Ar", age=30)


ValidationError: 1 validation error for Employee
name
  Value error, Name must contain at least 2 characters [type=value_error, input_value='Ar', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

<a id="112-multi-field-validation-model_validator"></a>
### 11.2 Multi-Field Validation (`@model_validator`)
Validating cross-field dependencies and multi-attribute rules across the complete model using `@model_validator(mode="after")` (e.g. confirming `password == confirm_password`).

In [90]:
# Model with cross-field validator (@model_validator mode='after')
class Signup(BaseModel):
    password: str
    confirm_password: str
    @model_validator(mode="after")
    def check_passwords(self):
        if (self.password!=self.confirm_password):
            raise ValueError("Passwords do not match")
        return self


In [93]:
# Valid signup: password and confirm_password match
Signup(
    password="hello123", 
    confirm_password="hello123"
)

Signup(password='hello123', confirm_password='hello123')

In [94]:
# Validation failure: passwords do not match (raises ValueError)
Signup(
    password="hello123", 
    confirm_password="wrong123"
)

ValidationError: 1 validation error for Signup
  Value error, Passwords do not match [type=value_error, input_value={'password': 'hello123', ...m_password': 'wrong123'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

#### Key Rule of Thumb:
- **Single Field Validation**: Use `@field_validator` (validates or transforms an individual attribute).
- **Cross-Field / Model Validation**: Use `@model_validator` (validates relationships between multiple fields or the whole object).

<a id="113-comparison-matrix-field_validator-vs-model_validator"></a>
### 11.3 Comparison Matrix: `@field_validator` vs `@model_validator` 

| Validator Type | Scope | Primary Use Case | Method Signature |
| :--- | :--- | :--- | :--- |
| **`@field_validator`** | Single field | String formatting, range checking, single-field transforms | `@classmethod def func(cls, value)` |
| **`@model_validator`** | Complete model | Cross-field validation (e.g., `password == confirm_password`) | `def func(self)` (mode='after') |

<a id="12-nested-schemas--complex-models"></a>
## 12. Nested Schemas & Complex Models

<a id="121-defining-nested-address--company-models"></a>
### 12.1 Defining Nested Address & Company Models
Composing complex hierarchical data models by embedding child Pydantic models (`Address` and `Company`) inside a parent Pydantic model (`User`).

```text
User (Parent Schema)
 ├── name: str
 ├── age: int
 ├── email: EmailStr
 ├── address: Address (Child Schema: city, state, pin_code)
 └── company: Company (Child Schema: company_name, department)
```

In [95]:
# Child schema: Address model
class Address(BaseModel):
    city: str
    state: str
    pin_code: int


In [96]:
# Child schema: Company model
class Company(BaseModel):
    company_name: str
    department: str


In [97]:
# Parent schema: User model composing Address and Company
class User(BaseModel):
    name: str
    age: int
    email: EmailStr
    address: Address
    company: Company


<a id="122-instantiating--validating-nested-payloads"></a>
### 12.2 Instantiating & Validating Nested Payloads
Pydantic automatically parses nested dictionaries into child model instances (`Address` and `Company`) during direct instantiation and when validating raw dictionary payloads with `User.model_validate(user_data)`.

In [98]:
# Direct instantiation with nested dictionary payloads (automatically parses child models)
User(
    name="Areeb",
    age=30,
    email="areeb@gmail.com",

    address={
        "city": "Bangalore",
        "state": "Karnataka",
        "pin_code": 560001
    },

    company={
        "company_name": "ABC Technologies",
        "department": "AI Engineering"
    }
)

User(name='Areeb', age=30, email='areeb@gmail.com', address=Address(city='Bangalore', state='Karnataka', pin_code=560001), company=Company(company_name='ABC Technologies', department='AI Engineering'))

In [99]:
# Raw nested dictionary payload
user_data = {
    "name": "Areeb",
    "age": 30,
    "email": "areeb@gmail.com",

    "address": {
        "city": "Bangalore",
        "state": "Karnataka",
        "pin_code": 560001
    },

    "company": {
        "company_name": "ABC Technologies",
        "department": "AI Engineering"
    }
}

In [100]:
# Validating complete nested dictionary payload with User.model_validate()
User.model_validate(user_data)

User(name='Areeb', age=30, email='areeb@gmail.com', address=Address(city='Bangalore', state='Karnataka', pin_code=560001), company=Company(company_name='ABC Technologies', department='AI Engineering'))